
# Direction S FIXED — External Salmonella new-drug data proof from BV-BRC/PATRIC

Notebook này lấy dữ liệu ngoài project Salmonella 2025 để kiểm tra:

> Có thuốc mới ngoài AMP/AUG/AXO/CHL/FOX không, và có đủ nhãn S/R để làm proof-of-concept không?

Nguồn chính: **BV-BRC / PATRIC AMR Metadata Review 2021**.

Lý do chọn nguồn này:
- có dữ liệu dạng `genome_id – antibiotic – phenotype`;
- có nhiều loài và nhiều thuốc;
- có thể lọc *Salmonella* và tìm các thuốc chưa có trong bộ hiện tại;
- không cần tải raw FASTQ trong bước đầu.

Notebook này làm:
1. Clone repo `BV-BRC/AMRMetadataReview_2021`.
2. Đọc các file tabular AMR nếu có.
3. Tự phát hiện cột genome/species/antibiotic/SIR.
4. Lọc *Salmonella*.
5. Tìm các thuốc mới có đủ mẫu S/R.
6. Tải PubChem descriptors cho các thuốc ứng viên nếu tìm được tên thuốc.
7. Xuất bảng candidate external new drugs.

Lưu ý:
- Đây là bước **data proof**. Nếu muốn train model external thật sự, cần thêm genome feature matrix tương ứng với `genome_id`.
- Repo BV-BRC có nói full model cần tải FASTA rất lớn, nên notebook này tránh tải raw genome.


## Bản FIXED sửa gì?

Bản trước nhận nhầm `taxon_id` là cột species nên lọc Salmonella ra 0 dòng.  
Bản này ưu tiên dùng `genome_name` / `genome name` để lọc *Salmonella*, sau đó mới fallback sang taxon/species khác.


In [1]:

# =========================
# 0. Import + config
# =========================

import os
import re
import json
import shutil
import warnings
from pathlib import Path
from collections import Counter, defaultdict

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from IPython.display import display

try:
    import requests
    HAS_REQUESTS = True
except Exception:
    HAS_REQUESTS = False

BASE_DIR = Path("/content/salmonella_direction_S_external_new_drug_data")
OUTPUT_DIR = BASE_DIR / "outputs"
REPO_DIR = BASE_DIR / "AMRMetadataReview_2021"

for d in [BASE_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CURRENT_DRUGS = {"AMP", "AUG", "AXO", "CHL", "FOX"}

# Từ khóa map tên thuốc hay gặp sang mã/nhóm. Không cần đầy đủ tuyệt đối.
ANTIBIOTIC_SYNONYMS = {
    "ampicillin": "AMP",
    "amoxicillin/clavulanic acid": "AUG",
    "amoxicillin-clavulanic acid": "AUG",
    "amoxicillin + clavulanic acid": "AUG",
    "ceftriaxone": "AXO",
    "chloramphenicol": "CHL",
    "cefoxitin": "FOX",
    "tetracycline": "TET",
    "ciprofloxacin": "CIP",
    "nalidixic acid": "NAL",
    "gentamicin": "GEN",
    "streptomycin": "STR",
    "sulfisoxazole": "FIS",
    "sulfamethoxazole": "SMX",
    "trimethoprim-sulfamethoxazole": "SXT",
    "trimethoprim/sulfamethoxazole": "SXT",
    "azithromycin": "AZM",
    "ceftazidime": "CAZ",
    "cefotaxime": "CTX",
    "kanamycin": "KAN",
}

MIN_S = 50
MIN_R = 50

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("HAS_REQUESTS:", HAS_REQUESTS)


BASE_DIR: /content/salmonella_direction_S_external_new_drug_data
OUTPUT_DIR: /content/salmonella_direction_S_external_new_drug_data/outputs
HAS_REQUESTS: True


In [2]:

# =========================
# 1. Clone BV-BRC/PATRIC AMR metadata repo
# =========================

if not REPO_DIR.exists():
    !git clone --depth 1 https://github.com/BV-BRC/AMRMetadataReview_2021.git "{REPO_DIR}"
else:
    print("Repo đã tồn tại:", REPO_DIR)

print("Repo files:")
!find "{REPO_DIR}" -maxdepth 3 -type f | sed 's#^#/##' | head -80


Cloning into '/content/salmonella_direction_S_external_new_drug_data/AMRMetadataReview_2021'...
remote: Enumerating objects: 102, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 102 (delta 23), reused 102 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (102/102), 559.36 MiB | 20.24 MiB/s, done.
Resolving deltas: 100% (23/23), done.
Updating files: 100% (95/95), done.
Repo files:
//content/salmonella_direction_S_external_new_drug_data/AMRMetadataReview_2021/.gitmodules
//content/salmonella_direction_S_external_new_drug_data/AMRMetadataReview_2021/README.md
//content/salmonella_direction_S_external_new_drug_data/AMRMetadataReview_2021/run.sh
//content/salmonella_direction_S_external_new_drug_data/AMRMetadataReview_2021/getAccBySpc.py
//content/salmonella_direction_S_external_new_drug_data/AMRMetadataReview_2021/tabular/README.md
//content/salmonella_direction_S_external_new_drug_data/AMRMetadataReview_2021/ta

In [3]:

# =========================
# 2. Tìm và đọc file tabular AMR
# =========================

tabular_dir = REPO_DIR / "tabular"
if not tabular_dir.exists():
    raise FileNotFoundError("Không thấy thư mục tabular trong repo")

files = sorted([p for p in tabular_dir.glob("*") if p.is_file()])
print("Tabular files:")
for p in files:
    print("-", p.name, round(p.stat().st_size/1024/1024, 2), "MB")

def read_any_table(path):
    path = Path(path)
    # BV-BRC tabular thường là tab-separated.
    try:
        df = pd.read_csv(path, sep="\t", low_memory=False)
        if df.shape[1] == 1:
            df = pd.read_csv(path, low_memory=False)
        return df
    except Exception:
        return pd.read_csv(path, low_memory=False)

tables = {}
for p in files:
    try:
        df = read_any_table(p)
        tables[p.name] = df
        print("\n", p.name, df.shape)
        display(df.head())
    except Exception as e:
        print("Không đọc được", p.name, e)


Tabular files:
- AMR.tbl.v4 61.28 MB
- README.md 0.0 MB
- amr.mic.filt.tab 4.37 MB
- amr.sir.filt.tab 8.48 MB

 AMR.tbl.v4 (550802, 17)


,genome_id,genome_name,taxon_id,antibiotic,resistant_phenotype,measurement,measurement_sign,measurement_value,measurement_unit,laboratory_typing_method,laboratory_typing_method_version,laboratory_typing_platform,vendor,testing_standard,testing_standard_year,source,Unnamed: 16
0,32002.4,Achromobacter denitrificans strain USDA-ARS-US...,32002,ampicillin,NaN,16,==,16,mg/L,mic,BOPO6F plate; cattle host,sensititre,trek diagnostic systems,clsi,NaN,NaN,NaN
1,32002.4,Achromobacter denitrificans strain USDA-ARS-US...,32002,ceftiofur,NaN,>8,>,8,mg/L,mic,BOPO6F plate; cattle host,sensititre,trek diagnostic systems,clsi,NaN,NaN,NaN
2,32002.4,Achromobacter denitrificans strain USDA-ARS-US...,32002,chlortetracycline,NaN,8,==,8,mg/L,mic,BOPO6F plate; cattle host,sensititre,trek diagnostic systems,clsi,NaN,NaN,NaN
3,32002.4,Achromobacter denitrificans strain USDA-ARS-US...,32002,clindamycin,NaN,>16,>,16,mg/L,mic,BOPO6F plate; cattle host,sensititre,trek diagnostic systems,clsi,NaN,NaN,NaN
4,32002.4,Achromobacter denitrificans strain USDA-ARS-US...,32002,danofloxacin,NaN,1,==,1,mg/L,mic,BOPO6F plate; cattle host,sensititre,trek diagnostic systems,clsi,NaN,NaN,NaN


Không đọc được README.md Error tokenizing data. C error: Expected 1 fields in line 4, saw 3


 amr.mic.filt.tab (211028, 3)


,573.13584,CRO:MIC,>32
0,470.75330,NIT:MIC,512
1,28901.84170,GEN:MIC,0.5
2,562.42829,TET:MIC,2
3,485.43970,CRO:MIC,0.002
4,165302.70000,TET:MIC,<=4



 amr.sir.filt.tab (389746, 3)


,573.13584,CRO:eucast,4
0,1.733158e+03,RIF:missing,4
1,2.890184e+04,GEN:clsi,1
2,1.448835e+06,INH:missing,1
3,1.280302e+03,CIP:eucast,4
4,1.773195e+03,RIF:who,1


In [4]:

# =========================
# 3. Hàm chuẩn hóa cột — FIXED species detection
# =========================

def norm_col(c):
    return re.sub(r"[^a-z0-9]+", "_", str(c).lower()).strip("_")

def find_col(df, keywords):
    cols = list(df.columns)
    norm_map = {c: norm_col(c) for c in cols}
    for c, nc in norm_map.items():
        if all(k in nc for k in keywords):
            return c
    return None

def find_col_any(df, list_of_keyword_lists):
    for kws in list_of_keyword_lists:
        c = find_col(df, kws)
        if c is not None:
            return c
    return None

def infer_columns(df):
    cols = {}
    cols["genome_id"] = find_col_any(df, [
        ["genome", "id"],
        ["genome_id"],
        ["genome"],
    ])

    # FIX: ưu tiên genome_name trước taxon_id.
    # AMR.tbl.v4 có genome_name chứa tên loài như "Salmonella enterica ...".
    cols["species"] = find_col_any(df, [
        ["genome", "name"],
        ["genome_name"],
        ["organism", "name"],
        ["organism"],
        ["species"],
        ["taxon", "name"],
        ["taxon"],
    ])

    cols["antibiotic"] = find_col_any(df, [
        ["antibiotic"],
        ["drug"],
        ["antimicrobial"],
    ])

    cols["phenotype"] = find_col_any(df, [
        ["resistant", "phenotype"],
        ["phenotype"],
        ["sir"],
        ["resistance"],
    ])

    cols["mic"] = find_col(df, ["mic"])
    return cols

def parse_sir_value(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in ["r", "resistant", "resistance", "non-susceptible", "nonsusceptible"]:
        return "R"
    if s in ["s", "susceptible", "sensitive", "susceptibility"]:
        return "S"
    if s in ["i", "intermediate"]:
        return "I"
    if "resistant" in s and "suscept" not in s:
        return "R"
    if "suscept" in s or "sensitive" in s:
        return "S"
    if "intermediate" in s:
        return "I"
    return np.nan

def normalize_antibiotic_name(x):
    s = str(x).strip()
    s = re.sub(r":.*$", "", s)
    s = s.replace("_", " ").replace("-", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def antibiotic_code(name):
    s = normalize_antibiotic_name(name).lower()
    if s in ANTIBIOTIC_SYNONYMS:
        return ANTIBIOTIC_SYNONYMS[s]
    for key, code in ANTIBIOTIC_SYNONYMS.items():
        if key in s:
            return code
    return re.sub(r"[^A-Za-z]", "", s).upper()[:3]


In [5]:

# =========================
# 4. Tạo bảng AMR long format từ các file có thể dùng
# =========================

long_tables = []

for name, df in tables.items():
    cols = infer_columns(df)
    print("\nFile:", name)
    print("Inferred columns:", cols)

    # Case 1: amr.sir.filt.tab thường có 3 cột: genome_id, antibiotic:breakpoint_def, SIR
    if df.shape[1] >= 3 and (cols["genome_id"] is None or cols["antibiotic"] is None or cols["phenotype"] is None):
        tmp = df.copy()
        tmp_cols = list(tmp.columns)
        tmp = tmp.rename(columns={
            tmp_cols[0]: "genome_id",
            tmp_cols[1]: "antibiotic_raw",
            tmp_cols[2]: "phenotype_raw"
        })
        tmp["source_file"] = name
        tmp["antibiotic"] = tmp["antibiotic_raw"].apply(normalize_antibiotic_name)
        tmp["phenotype"] = tmp["phenotype_raw"].apply(parse_sir_value)
        # species chưa có, để NaN
        tmp["species"] = np.nan
        long_tables.append(tmp[["source_file", "genome_id", "species", "antibiotic", "phenotype"]])
        continue

    if cols["genome_id"] and cols["antibiotic"] and cols["phenotype"]:
        tmp = df.copy()
        tmp["source_file"] = name
        tmp["genome_id"] = tmp[cols["genome_id"]].astype(str)
        tmp["species"] = tmp[cols["species"]].astype(str) if cols["species"] else np.nan
        tmp["antibiotic"] = tmp[cols["antibiotic"]].apply(normalize_antibiotic_name)
        tmp["phenotype"] = tmp[cols["phenotype"]].apply(parse_sir_value)
        long_tables.append(tmp[["source_file", "genome_id", "species", "antibiotic", "phenotype"]])

if not long_tables:
    raise RuntimeError("Không tạo được AMR long table từ tabular files")

amr_long = pd.concat(long_tables, ignore_index=True)
amr_long = amr_long.dropna(subset=["genome_id", "antibiotic", "phenotype"])
amr_long = amr_long[amr_long["phenotype"].isin(["S", "R"])].copy()
amr_long["drug_code"] = amr_long["antibiotic"].apply(antibiotic_code)

print("amr_long:", amr_long.shape)
display(amr_long.head())

amr_long.to_csv(OUTPUT_DIR / "external_bvbrc_amr_long_all.csv", index=False)



File: AMR.tbl.v4
Inferred columns: {'genome_id': 'genome_id', 'species': 'genome_name', 'antibiotic': 'antibiotic', 'phenotype': 'resistant_phenotype', 'mic': None}

File: amr.mic.filt.tab
Inferred columns: {'genome_id': None, 'species': None, 'antibiotic': None, 'phenotype': None, 'mic': 'CRO:MIC'}

File: amr.sir.filt.tab
Inferred columns: {'genome_id': None, 'species': None, 'antibiotic': None, 'phenotype': None, 'mic': None}
amr_long: (348230, 6)


,source_file,genome_id,species,antibiotic,phenotype,drug_code
19,AMR.tbl.v4,134375.5,Achromobacter sp. strain 34,amikacin,S,AMI
21,AMR.tbl.v4,134375.5,Achromobacter sp. strain 34,cefepime,S,CEF
22,AMR.tbl.v4,134375.5,Achromobacter sp. strain 34,cefepime,R,CEF
23,AMR.tbl.v4,134375.5,Achromobacter sp. strain 34,ceftazidime,R,CAZ
24,AMR.tbl.v4,134375.5,Achromobacter sp. strain 34,ceftazidime,S,CAZ


In [6]:

# =========================
# 5. Lọc Salmonella — FIXED
# =========================

# Kiểm tra cột species/genome_name sau khi infer.
print("Ví dụ species/genome_name trong AMR long:")
display(amr_long[["source_file", "genome_id", "species", "antibiotic", "phenotype"]].head(10))

# Lọc Salmonella bằng text trong species/genome_name.
species_text = amr_long["species"].astype(str)
salm = amr_long[species_text.str.contains("salmonella", case=False, na=False)].copy()

print("Salmonella rows after fixed filter:", salm.shape)

if len(salm) > 0:
    target_df = salm.copy()
    target_name = "Salmonella"
else:
    print("Vẫn không tìm được Salmonella bằng species/genome_name.")
    print("Fallback: tạo summary toàn bộ species, nhưng KHÔNG dùng làm proof Salmonella.")
    target_df = amr_long.copy()
    target_name = "all_species"

summary = (
    target_df
    .groupby(["antibiotic", "drug_code", "phenotype"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ["S", "R"]:
    if col not in summary.columns:
        summary[col] = 0

summary["n_total_SR"] = summary["S"] + summary["R"]
summary["resistant_rate"] = summary["R"] / summary["n_total_SR"].replace(0, np.nan)
summary["is_current_drug"] = summary["drug_code"].isin(CURRENT_DRUGS)

summary = summary.sort_values(["is_current_drug", "n_total_SR"], ascending=[True, False])

display(summary.head(30))
summary.to_csv(OUTPUT_DIR / f"external_bvbrc_{target_name}_drug_summary.csv", index=False)

if target_name == "Salmonella":
    print("OK: Summary này là Salmonella-specific.")
else:
    print("CẢNH BÁO: Summary này là all-species, chưa phải Salmonella-specific.")


Ví dụ species/genome_name trong AMR long:


,source_file,genome_id,species,antibiotic,phenotype
19,AMR.tbl.v4,134375.5,Achromobacter sp. strain 34,amikacin,S
21,AMR.tbl.v4,134375.5,Achromobacter sp. strain 34,cefepime,S
22,AMR.tbl.v4,134375.5,Achromobacter sp. strain 34,cefepime,R
23,AMR.tbl.v4,134375.5,Achromobacter sp. strain 34,ceftazidime,R
24,AMR.tbl.v4,134375.5,Achromobacter sp. strain 34,ceftazidime,S
25,AMR.tbl.v4,134375.5,Achromobacter sp. strain 34,ceftriaxone,R
26,AMR.tbl.v4,134375.5,Achromobacter sp. strain 34,ciprofloxacin,R
27,AMR.tbl.v4,134375.5,Achromobacter sp. strain 34,ciprofloxacin,R
28,AMR.tbl.v4,134375.5,Achromobacter sp. strain 34,colistin,S
30,AMR.tbl.v4,134375.5,Achromobacter sp. strain 34,gentamicin,S


Salmonella rows after fixed filter: (45517, 6)


phenotype,antibiotic,drug_code,R,S,n_total_SR,resistant_rate,is_current_drug
17,ciprofloxacin,CIP,627,2735,3362,0.186496,False
43,trimethoprim sulphamethoxazole,TRI,375,2765,3140,0.119427,False
39,tetracycline,TET,1611,1358,2969,0.542607,False
28,nalidixic acid,NAL,260,2650,2910,0.089347,False
34,streptomycin,STR,1257,1650,2907,0.432405,False
23,gentamicin,GEN,314,2358,2672,0.117515,False
5,azithromycin,AZM,49,2289,2338,0.020958,False
1,amoxicillin clavulanic acid,AMO,401,1883,2284,0.175569,False
13,ceftiofur,CEF,400,1863,2263,0.176757,False
36,sulfisoxazole,FIS,754,1379,2133,0.353493,False


OK: Summary này là Salmonella-specific.


In [7]:

# =========================
# 6. Chọn candidate thuốc mới
# =========================

candidates = summary[
    (~summary["is_current_drug"]) &
    (summary["S"] >= MIN_S) &
    (summary["R"] >= MIN_R)
].copy()

candidates = candidates.sort_values(["n_total_SR", "R"], ascending=False)
display(candidates.head(30))

candidates.to_csv(OUTPUT_DIR / f"external_new_drug_candidates_{target_name}.csv", index=False)

if len(candidates) == 0:
    print("Không tìm được thuốc mới đủ S/R theo ngưỡng hiện tại.")
    print("Có thể giảm MIN_S/MIN_R hoặc cần species mapping để lọc đúng Salmonella.")
else:
    print("Top candidate thuốc mới:")
    for _, row in candidates.head(10).iterrows():
        print("-", row["antibiotic"], "| code:", row["drug_code"], "| S:", int(row["S"]), "| R:", int(row["R"]))


phenotype,antibiotic,drug_code,R,S,n_total_SR,resistant_rate,is_current_drug
17,ciprofloxacin,CIP,627,2735,3362,0.186496,False
43,trimethoprim sulphamethoxazole,TRI,375,2765,3140,0.119427,False
39,tetracycline,TET,1611,1358,2969,0.542607,False
28,nalidixic acid,NAL,260,2650,2910,0.089347,False
34,streptomycin,STR,1257,1650,2907,0.432405,False
23,gentamicin,GEN,314,2358,2672,0.117515,False
1,amoxicillin clavulanic acid,AMO,401,1883,2284,0.175569,False
13,ceftiofur,CEF,400,1863,2263,0.176757,False
36,sulfisoxazole,FIS,754,1379,2133,0.353493,False
25,kanamycin,KAN,80,1027,1107,0.072267,False


Top candidate thuốc mới:
- ciprofloxacin | code: CIP | S: 2735 | R: 627
- trimethoprim sulphamethoxazole | code: TRI | S: 2765 | R: 375
- tetracycline | code: TET | S: 1358 | R: 1611
- nalidixic acid | code: NAL | S: 2650 | R: 260
- streptomycin | code: STR | S: 1650 | R: 1257
- gentamicin | code: GEN | S: 2358 | R: 314
- amoxicillin clavulanic acid | code: AMO | S: 1883 | R: 401
- ceftiofur | code: CEF | S: 1863 | R: 400
- sulfisoxazole | code: FIS | S: 1379 | R: 754
- kanamycin | code: KAN | S: 1027 | R: 80


In [8]:

# =========================
# 7. PubChem descriptors cho candidate thuốc mới
# =========================

PUBCHEM_PROPERTIES = [
    "MolecularFormula",
    "MolecularWeight",
    "CanonicalSMILES",
    "XLogP",
    "TPSA",
    "HBondDonorCount",
    "HBondAcceptorCount",
    "RotatableBondCount",
    "HeavyAtomCount",
    "Complexity",
    "Charge",
]

def fetch_pubchem_by_name(name):
    if not HAS_REQUESTS:
        return None
    prop_str = ",".join(PUBCHEM_PROPERTIES)
    safe_name = requests.utils.quote(str(name))
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{safe_name}/property/{prop_str}/JSON"
    try:
        r = requests.get(url, timeout=20)
        if r.status_code != 200:
            return None
        data = r.json()
        props = data["PropertyTable"]["Properties"][0]
        return props
    except Exception:
        return None

pubchem_rows = []
for _, row in candidates.head(20).iterrows():
    props = fetch_pubchem_by_name(row["antibiotic"])
    if props is None:
        props = {}
    props["antibiotic"] = row["antibiotic"]
    props["drug_code"] = row["drug_code"]
    props["S"] = row["S"]
    props["R"] = row["R"]
    props["n_total_SR"] = row["n_total_SR"]
    pubchem_rows.append(props)

candidate_pubchem = pd.DataFrame(pubchem_rows)
display(candidate_pubchem)

candidate_pubchem.to_csv(OUTPUT_DIR / "external_candidate_new_drug_pubchem.csv", index=False)


,CID,MolecularFormula,MolecularWeight,ConnectivitySMILES,XLogP,TPSA,Complexity,Charge,HBondDonorCount,HBondAcceptorCount,RotatableBondCount,HeavyAtomCount,antibiotic,drug_code,S,R,n_total_SR
0,2764.0,C17H18FN3O3,331.34,C1CC1N2C=C(C(=O)C3=CC(=C(C=C32)N4CCNCC4)F)C(=O)O,-1.1,72.9,571.0,0.0,2.0,7.0,3.0,24.0,ciprofloxacin,CIP,2735,627,3362
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,trimethoprim sulphamethoxazole,TRI,2765,375,3140
2,54675776.0,C22H24N2O8,444.4,CC1(C2CC3C(C(=O)C(=C(C3(C(=O)C2=C(C4=C1C=CC=C4...,-2.0,182.0,971.0,0.0,6.0,9.0,2.0,32.0,tetracycline,TET,1358,1611,2969
3,4421.0,C12H12N2O3,232.23,CCN1C=C(C(=O)C2=C1N=C(C=C2)C)C(=O)O,1.4,70.5,378.0,0.0,1.0,5.0,2.0,17.0,nalidixic acid,NAL,2650,260,2910
4,19649.0,C21H39N7O12,581.6,CC1C(C(C(O1)OC2C(C(C(C(C2O)O)N=C(N)N)O)N=C(N)N...,-8.0,336.0,940.0,0.0,12.0,15.0,9.0,40.0,streptomycin,STR,1650,1257,2907
5,3467.0,C21H43N5O7,477.6,CC(C1CCC(C(O1)OC2C(CC(C(C2O)OC3C(C(C(CO3)(C)O)...,-4.1,200.0,636.0,0.0,8.0,12.0,7.0,33.0,gentamicin,GEN,2358,314,2672
6,23665637.0,C24H27KN4O10S,602.7,CC1(C(N2C(S1)C(C2=O)NC(=O)C(C3=CC=C(C=C3)O)N)C...,NaN,248.0,919.0,0.0,5.0,12.0,6.0,40.0,amoxicillin clavulanic acid,AMO,1883,401,2284
7,6328657.0,C19H17N5O7S3,523.6,CON=C(C1=CSC(=N1)N)C(=O)NC2C3N(C2=O)C(=C(CS3)C...,0.2,256.0,945.0,0.0,3.0,13.0,9.0,34.0,ceftiofur,CEF,1863,400,2263
8,5344.0,C11H13N3O3S,267.31,CC1=C(ON=C1C)NS(=O)(=O)C2=CC=C(C=C2)N,1.0,107.0,373.0,0.0,2.0,6.0,3.0,18.0,sulfisoxazole,FIS,1379,754,2133
9,6032.0,C18H36N4O11,484.5,C1C(C(C(C(C1N)OC2C(C(C(C(O2)CN)O)O)O)O)OC3C(C(...,-6.9,283.0,638.0,0.0,11.0,15.0,6.0,33.0,kanamycin,KAN,1027,80,1107


In [9]:

# =========================
# 8. Nếu species thiếu: thử tải PATRIC/BV-BRC AMR file chính từ FTP/HTTP
# =========================

# Một số nguồn nêu file PATRIC_genomes_AMR.txt trên BV-BRC FTP.
# Cell này thử tải nhẹ bằng https/ftp. Nếu không được thì bỏ qua.

extra_urls = [
    "https://ftp.bvbrc.org/RELEASE_NOTES/PATRIC_genomes_AMR.txt",
    "ftp://ftp.bvbrc.org/RELEASE_NOTES/PATRIC_genomes_AMR.txt",
]

extra_path = BASE_DIR / "PATRIC_genomes_AMR.txt"

if not extra_path.exists():
    for url in extra_urls:
        print("Trying:", url)
        ret = os.system(f'wget -q -O "{extra_path}" "{url}"')
        if ret == 0 and extra_path.exists() and extra_path.stat().st_size > 1000:
            print("Downloaded:", extra_path, "size MB", round(extra_path.stat().st_size/1024/1024, 2))
            break
        else:
            if extra_path.exists():
                extra_path.unlink()

if extra_path.exists():
    try:
        patric_amr = pd.read_csv(extra_path, sep="\t", low_memory=False)
        print("PATRIC_genomes_AMR:", patric_amr.shape)
        display(patric_amr.head())
        patric_amr.to_csv(OUTPUT_DIR / "PATRIC_genomes_AMR_head_check.csv", index=False)
    except Exception as e:
        print("Không đọc được PATRIC_genomes_AMR:", e)
else:
    print("Không tải được PATRIC_genomes_AMR qua notebook này.")


Trying: https://ftp.bvbrc.org/RELEASE_NOTES/PATRIC_genomes_AMR.txt
Trying: ftp://ftp.bvbrc.org/RELEASE_NOTES/PATRIC_genomes_AMR.txt
Không tải được PATRIC_genomes_AMR qua notebook này.


In [10]:

# =========================
# 9. Auto conclusion
# =========================

lines = []
lines.append("# Direction S — External new-drug data proof")
lines.append("")
lines.append("## 1. Nguồn dữ liệu")
lines.append("Notebook dùng BV-BRC/PATRIC AMR Metadata Review 2021 để tìm thuốc ngoài AMP/AUG/AXO/CHL/FOX. Bản FIXED ưu tiên lọc Salmonella bằng genome_name thay vì taxon_id.")
lines.append("")
lines.append("## 2. Kết quả dữ liệu")

lines.append(f"- Tổng số dòng AMR long format: {len(amr_long)}.")
if target_name == "Salmonella":
    lines.append(f"- Lọc được Salmonella-specific rows: {len(salm)}.")
else:
    lines.append("- Chưa lọc được Salmonella-specific rows; kết quả candidate là all-species fallback, không dùng làm kết luận Salmonella.")

lines.append(f"- Số candidate thuốc mới đủ ngưỡng S>={MIN_S}, R>={MIN_R}: {len(candidates)}.")

if len(candidates) > 0:
    lines.append("")
    lines.append("## 3. Candidate thuốc mới")
    for _, row in candidates.head(10).iterrows():
        lines.append(
            f"- {row['antibiotic']} ({row['drug_code']}): "
            f"S={int(row['S'])}, R={int(row['R'])}, total={int(row['n_total_SR'])}, "
            f"resistant_rate={row['resistant_rate']:.3f}."
        )

lines.append("")
lines.append("## 4. Diễn giải")
lines.append("- Nếu có candidate đủ S/R, có thể dùng làm thuốc mới external để test hướng genome–drug generalization.")
lines.append("- Nếu chỉ có phenotype mà chưa có genome feature matrix, bước tiếp theo là lấy genome features tương ứng với genome_id từ BV-BRC/PATRIC hoặc dùng pipeline đã được repo cung cấp.")
lines.append("- Không nên tải full FASTA nếu mục tiêu hiện tại là proof nhẹ, vì repo BV-BRC nói full run có thể cần dữ liệu rất lớn.")

conclusion = "\n".join(lines)
print(conclusion)

with open(OUTPUT_DIR / "AUTO_CONCLUSION_DIRECTION_S.md", "w", encoding="utf-8") as f:
    f.write(conclusion)


# Direction S — External new-drug data proof

## 1. Nguồn dữ liệu
Notebook dùng BV-BRC/PATRIC AMR Metadata Review 2021 để tìm thuốc ngoài AMP/AUG/AXO/CHL/FOX. Bản FIXED ưu tiên lọc Salmonella bằng genome_name thay vì taxon_id.

## 2. Kết quả dữ liệu
- Tổng số dòng AMR long format: 348230.
- Lọc được Salmonella-specific rows: 45517.
- Số candidate thuốc mới đủ ngưỡng S>=50, R>=50: 13.

## 3. Candidate thuốc mới
- ciprofloxacin (CIP): S=2735, R=627, total=3362, resistant_rate=0.186.
- trimethoprim sulphamethoxazole (TRI): S=2765, R=375, total=3140, resistant_rate=0.119.
- tetracycline (TET): S=1358, R=1611, total=2969, resistant_rate=0.543.
- nalidixic acid (NAL): S=2650, R=260, total=2910, resistant_rate=0.089.
- streptomycin (STR): S=1650, R=1257, total=2907, resistant_rate=0.432.
- gentamicin (GEN): S=2358, R=314, total=2672, resistant_rate=0.118.
- amoxicillin clavulanic acid (AMO): S=1883, R=401, total=2284, resistant_rate=0.176.
- ceftiofur (CEF): S=1863, R=400, total=2263, resista

In [11]:

# =========================
# 10. Zip outputs
# =========================

zip_path = BASE_DIR / "salmonella_direction_S_external_new_drug_data_outputs.zip"
if zip_path.exists():
    zip_path.unlink()

shutil.make_archive(str(zip_path).replace(".zip", ""), "zip", OUTPUT_DIR)

print("Output dir:", OUTPUT_DIR)
print("Zip:", zip_path)

# Nếu muốn tải về:
# from google.colab import files
# files.download(str(zip_path))


Output dir: /content/salmonella_direction_S_external_new_drug_data/outputs
Zip: /content/salmonella_direction_S_external_new_drug_data/salmonella_direction_S_external_new_drug_data_outputs.zip
